# 10. Entity Analysis: Trends, Associations, and Networks

This notebook analyses the entity dataset built in `09_entity_dataset.ipynb`. It loads the saved tables (per-article entities, frequencies, timelines, and co-occurrence) and turns them into interpretable analysis: which diseases and chemicals dominate, what is rising or falling, which drug-disease associations are strongest, and how diseases cluster into a co-occurrence network.

It reads from `data/3_entities/`, so it never touches the 3 million abstracts. It works on whichever method's tables are present (dictionary always, model if notebook 09's model pass was run); set METHOD below to choose.

## Setup: load the saved entity tables

- Loads `disease_frequency_<METHOD>.parquet`, `chemical_frequency_<METHOD>.parquet`, `disease_timeline_<METHOD>.parquet`, `chemical_timeline_<METHOD>.parquet`, `disease_chemical_pairs_<METHOD>.parquet`, `disease_disease_pairs_<METHOD>.parquet` from `data/3_entities/`.
- Loads per-article table `article_entities_dictionary.parquet` or `article_entities_{METHOD}.parquet`.
- Prints status lines showing which tables were found and basic counts (distinct diseases, chemicals, pairs, articles).

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ROOT = os.path.dirname(os.getcwd())
ENT_DIR = os.path.join(ROOT, "data", "3_entities")

# choose which method's tables to analyse: "dictionary" (always) or a model tag like "model_full" / "model_50000"
METHOD = "dictionary"

def load(name):
    path = os.path.join(ENT_DIR, f"{name}_{METHOD}.parquet")
    if not os.path.exists(path):
        print(f"missing: {path}")
        return None
    return pd.read_parquet(path)

disease_freq = load("disease_frequency")
chemical_freq = load("chemical_frequency")
disease_tl = load("disease_timeline")
chemical_tl = load("chemical_timeline")
dc_pairs = load("disease_chemical_pairs")
dd_pairs = load("disease_disease_pairs")

# per-article table (filename pattern differs slightly)
_art = os.path.join(ENT_DIR, f"article_entities_{'dictionary' if METHOD=='dictionary' else METHOD}.parquet")
article_df = pd.read_parquet(_art) if os.path.exists(_art) else None

print(f"analysing METHOD = {METHOD}")
print(f"  diseases ranked: {len(disease_freq):,}" if disease_freq is not None else "  no disease freq")
print(f"  chemicals ranked: {len(chemical_freq):,}" if chemical_freq is not None else "  no chemical freq")
print(f"  disease-chemical pairs: {len(dc_pairs):,}" if dc_pairs is not None else "")
print(f"  articles: {len(article_df):,}" if article_df is not None else "")

## Cleaning: merge duplicates and drop noise

- Merges simple plurals (e.g., `tumor`/`tumors`) by summing counts.
- Drops explicit non-entity noise lists for diseases and chemicals (e.g., `death`, `pain`, `smoking`, `alcohol`), with configurable lists.
- Keeps short, valid biomedical abbreviations (e.g., `ad`, `hf`, `ra`) while removing symbol/statistics noise (`±`, `ci`, `or`).
- Cleans frequency tables at analysis time so existing outputs are fixed without re-running extraction.

In [ ]:
# Clean the loaded tables: merge plural duplicates and drop non-disease/non-chemical noise.
# This runs at analysis time, so it works on existing model output without re-running the model.

NON_DISEASE = {
    "death", "deaths", "pain", "trauma", "bleeding", "fatigue", "toxicity", "psychiatric",
    "disability", "weight loss", "comorbidity", "malignancy", "depressive", "injuries",
    "injury", "complications", "complication", "symptoms", "symptom", "disease", "diseases",
    "disorder", "disorders", "syndrome", "syndromes", "lesion", "lesions", "abnormalities",
}
NON_CHEMICAL = {
    "smoking", "alcohol", "oxygen", "calcium", "sodium", "iron", "atp", "snp", "water",
    "amino acid", "amino acids", "nucleotide", "nucleotides", "protein", "proteins",
    "glucose",   # keep? glucose is borderline; comment out this line to keep it
}

def merge_plurals(freq_df, key):
    """Merge simple plural duplicates (X and Xs -> X) by summing their counts."""
    counts = dict(zip(freq_df[key], freq_df["n_articles"]))
    merged = {}
    for term, c in counts.items():
        base = term[:-1] if (term.endswith("s") and term[:-1] in counts) else term
        merged[base] = merged.get(base, 0) + c
    out = pd.DataFrame(merged.items(), columns=[key, "n_articles"])
    return out.sort_values("n_articles", ascending=False).reset_index(drop=True)

# Real 2-character disease/chemical abbreviations to keep (not noise).
KEEP_SHORT = {"ra", "ms", "tb", "mi", "dm", "ad", "hf", "ckd", "copd", "hiv", "ckd"}
SYMBOL_NOISE = {"±", "vs", "ci", "or", "rr", "hr", "sd", "se", "iqr", "+", "-", "="}

def clean_freq(freq_df, key, drop_set):
    if freq_df is None:
        return None
    f = freq_df[~freq_df[key].isin(drop_set | SYMBOL_NOISE)].copy()
    # keep terms that are length > 2, OR are recognised short abbreviations
    f = f[(f[key].str.len() > 2) | (f[key].isin(KEEP_SHORT))]
    f = f[f[key].str.contains("[a-z]", regex=True)]   # must contain a letter
    f = merge_plurals(f, key)
    return f

if disease_freq is not None:
    _before = len(disease_freq)
    disease_freq = clean_freq(disease_freq, "disease", NON_DISEASE)
    print(f"diseases: {_before:,} -> {len(disease_freq):,} after cleaning + plural merge")
    print("top 15 cleaned diseases:")
    print(disease_freq.head(15).to_string(index=False))

if chemical_freq is not None:
    _before = len(chemical_freq)
    chemical_freq = clean_freq(chemical_freq, "chemical", NON_CHEMICAL)
    print(f"\nchemicals: {_before:,} -> {len(chemical_freq):,} after cleaning")
    print("top 15 cleaned chemicals:")
    print(chemical_freq.head(15).to_string(index=False))

## 1. The most-studied diseases and chemicals

A simple but informative starting point: what the corpus studies most, by article count. The bar charts show the top 15 diseases and top 15 chemicals, giving a quick read on the corpus's headline research focus.

In [ ]:
if disease_freq is not None and chemical_freq is not None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    top_d = disease_freq.head(15)[::-1]
    sns.barplot(x="n_articles", y="disease", data=top_d, color="#c0392b", ax=axes[0])
    axes[0].set_title("Most-studied diseases (articles)"); axes[0].set_xlabel("articles"); axes[0].set_ylabel("")
    top_c = chemical_freq.head(15)[::-1]
    sns.barplot(x="n_articles", y="chemical", data=top_c, color="#2471a3", ax=axes[1])
    axes[1].set_title("Most-studied chemicals (articles)"); axes[1].set_xlabel("articles"); axes[1].set_ylabel("")
    plt.tight_layout(); plt.show()

**What this shows.** Cancer dominates the disease list with 272,442 articles, followed by infection (169,153) and diabetes (95,802). On the chemical side, insulin leads with 45,714 articles, followed by cholesterol (30,524) and estrogen (21,700). The bar charts visualise the top 15 for each category. Counts are article-level, so a single article can mention multiple entities.

## 2. What is rising and what is falling

- Uses per-year timelines to compute early-period vs late-period prevalence for each entity.
- Produces ranked lists and barplots for the fastest risers and fastest decliners (by late/early ratio, with minimum early presence threshold).
- Interpretation: risers often include pandemic-related or recently emergent topics; fallers typically reflect relative declines as research attention shifts.

In [ ]:
if disease_tl is not None:
    years = sorted(disease_tl.index)
    art_per_year = disease_tl.sum(axis=1)   # approx; for share use article counts if available
    early = [y for y in years if y <= years[0] + 2]
    late = [y for y in years if y >= years[-1] - 2]

    def trend_table(tl):
        rate_early = tl.loc[early].sum() / max(len(early), 1)
        rate_late = tl.loc[late].sum() / max(len(late), 1)
        out = pd.DataFrame({"early": rate_early, "late": rate_late})
        out["change"] = out["late"] - out["early"]
        out["ratio"] = (out["late"] + 1) / (out["early"] + 1)
        return out

    dz_trend = trend_table(disease_tl)
    risers = dz_trend[dz_trend["early"] >= 20].sort_values("ratio", ascending=False).head(10)
    fallers = dz_trend[dz_trend["early"] >= 20].sort_values("ratio").head(10)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(x=risers["ratio"], y=risers.index, color="#27ae60", ax=axes[0])
    axes[0].set_title("Fastest-rising diseases (late/early ratio)"); axes[0].set_xlabel("growth ratio"); axes[0].set_ylabel("")
    sns.barplot(x=fallers["ratio"], y=fallers.index, color="#7f8c8d", ax=axes[1])
    axes[1].set_title("Fastest-declining diseases"); axes[1].set_xlabel("growth ratio"); axes[1].set_ylabel("")
    plt.tight_layout(); plt.show()
    print("top risers:", ", ".join(risers.index[:8]))

**What this shows.** The trend table computes a late/early prevalence ratio for each disease present in both periods above a minimum-count threshold. The fastest risers are obesity (ratio about 10), ptsd, and atrial fibrillation, followed by cardiovascular disease, inflammation, dementia, influenza, and parkinson. The slowest growers, shown in the declining panel, are leukemia, myocardial infarction, hiv, schizophrenia, pneumonia, and tuberculosis. Note that every disease in the declining panel still has a ratio above 1, so these are relative slowdowns in attention rather than absolute drops, and covid-19 is excluded here because it has no early-period baseline to form a ratio. These are shifts in research attention, not disease incidence.

## 3. Strongest drug–disease associations

- Builds a heatmap of co-occurrence counts between the top diseases and top chemicals (e.g., top 12 each).
- Prints the top disease–chemical pairs overall (by `n_articles`).
- Interpretation: strong cells correspond to well-established therapeutic or measurement relationships (e.g., diabetes + insulin); co-occurrence is association, not evidence of treatment.

In [ ]:
if dc_pairs is not None and disease_freq is not None and chemical_freq is not None:
    top_dz = disease_freq.head(12)["disease"].tolist()
    top_ch = chemical_freq.head(12)["chemical"].tolist()
    sub = dc_pairs[dc_pairs["disease"].isin(top_dz) & dc_pairs["chemical"].isin(top_ch)]
    mat = sub.pivot_table(index="disease", columns="chemical", values="n_articles", fill_value=0)
    mat = mat.reindex(index=[d for d in top_dz if d in mat.index],
                      columns=[c for c in top_ch if c in mat.columns])
    plt.figure(figsize=(12, 9))
    sns.heatmap(mat, cmap="YlOrRd", annot=True, fmt=".0f", cbar_kws={"label": "co-mentioning articles"})
    plt.title("Drug-disease co-occurrence (top entities)")
    plt.xlabel("chemical"); plt.ylabel("disease")
    plt.tight_layout(); plt.show()

    print("strongest associations overall:")
    for _, r in dc_pairs.head(12).iterrows():
        print(f"  {r['n_articles']:>7,}   {r['disease']}  +  {r['chemical']}")

**What this shows.** The heatmap highlights the strongest co‑mention pairs among the top 12 diseases and chemicals. Diabetes + insulin (21,858) and diabetes + glucose (21,626) are the largest cells, followed by breast cancer + estrogen (8,670) and obesity + insulin (7,956). The colour intensity makes well‑established relationships immediately visible. Co‑occurrence is association from co‑mention in the same abstract, not evidence of a treatment or causal link.

## 4. Disease co-occurrence network (comorbidity map)

- Builds a disease–disease graph from `disease_disease_pairs_<METHOD>.parquet` where edges weight = co-mention counts.
- Filters by strong pairs (e.g., 85th percentile or absolute minimum), restricts to top diseases, finds the largest connected component, and runs modularity-based community detection.
- Visualizes communities with node sizes proportional to degree (weighted) and edge widths proportional to co-mention strength.
- Interpretation: communities group diseases often studied together (metabolic, cardiovascular, etc.); again, co-mention ≠ clinical comorbidity confirmation.

In [ ]:
try:
    import networkx as nx
except ImportError:
    print("networkx not installed; run pip install networkx")
    nx = None

if nx is not None and dd_pairs is not None and disease_freq is not None:
    MIN_PAIR = max(50, int(dd_pairs["n_articles"].quantile(0.85)))
    top_dz = set(disease_freq.head(40)["disease"])
    G = nx.Graph()
    for _, r in dd_pairs.iterrows():
        if r["disease_a"] in top_dz and r["disease_b"] in top_dz and r["n_articles"] >= MIN_PAIR:
            G.add_edge(r["disease_a"], r["disease_b"], weight=r["n_articles"])
    if G.number_of_nodes():
        comps = sorted(nx.connected_components(G), key=len, reverse=True)
        G = G.subgraph(comps[0]).copy()
        from networkx.algorithms.community import greedy_modularity_communities
        comms = list(greedy_modularity_communities(G, weight="weight"))
        cmap = {n: i for i, com in enumerate(comms) for n in com}
        palette = plt.colormaps["tab10"]
        colors = [palette(cmap[n] % 10) for n in G.nodes()]
        deg = dict(G.degree(weight="weight"))
        sizes = [300 + 0.05 * deg[n] for n in G.nodes()]
        pos = nx.spring_layout(G, k=0.6, seed=42, weight="weight")
        ws = [G[u][v]["weight"] for u, v in G.edges()]; wmax = max(ws) if ws else 1
        widths = [0.3 + 3 * (w / wmax) for w in ws]

        plt.figure(figsize=(14, 11))
        nx.draw_networkx_edges(G, pos, width=widths, alpha=0.2, edge_color="#888")
        nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, alpha=0.9)
        nx.draw_networkx_labels(G, pos, font_size=8)
        plt.title("Disease co-occurrence network (comorbidity communities)")
        plt.axis("off"); plt.tight_layout(); plt.show()
        print(f"network: {G.number_of_nodes()} diseases, {G.number_of_edges()} edges, {len(comms)} communities")
        for i, com in enumerate(comms[:6]):
            print(f"  community {i+1}: {', '.join(sorted(com)[:8])}")

**What this shows.** The network connects diseases frequently mentioned together, with edges weighted by co-mention count. After filtering to strong pairs and the top diseases, the largest connected component has 35 nodes and 100 edges, and modularity-based detection splits it into 5 communities: an infectious and inflammatory cluster (infection, inflammation, hiv, covid-19, asthma, arthritis, fibrosis), a cardiometabolic cluster (diabetes, obesity, hypertension, heart failure, myocardial infarction, stroke, atrial fibrillation, cardiovascular disease), an oncology cluster (cancer, breast cancer, prostate cancer, lung cancer, leukemia, melanoma), a mental-health cluster (depression, anxiety, ptsd, schizophrenia), and a neurodegenerative cluster (alzheimer's disease, dementia, parkinson). Note that metabolic and cardiovascular conditions fall in the same community here, not separate ones. Node size reflects total co-occurrence strength. This is a data-driven comorbidity map based on co-mention in abstracts, not clinical diagnosis data.

## 5. Entity richness per article

- Uses the per-article table to compute mean `n_diseases` and mean `n_chemicals` per article by year, and the percent of articles that mention at least one listed disease.
- Plots trends over time: mean entities per article and coverage (% of articles mentioning a disease).
- Interpretation: rising mean entities may indicate broader study scope or better detection; coverage is bounded by the term lists and abstract content.

In [ ]:
if article_df is not None:
    dz_col = "dz_dict" if "dz_dict" in article_df else ("dz_model" if "dz_model" in article_df else None)
    if dz_col and "n_diseases" in article_df:
        by_year = article_df.groupby("year").agg(
            mean_diseases=("n_diseases", "mean"),
            mean_chemicals=("n_chemicals", "mean"),
            pct_with_disease=("n_diseases", lambda s: (s > 0).mean() * 100),
        )
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        axes[0].plot(by_year.index, by_year["mean_diseases"], marker="o", label="diseases")
        axes[0].plot(by_year.index, by_year["mean_chemicals"], marker="s", label="chemicals")
        axes[0].set_title("Mean entities per article"); axes[0].set_xlabel("year"); axes[0].legend()
        axes[1].plot(by_year.index, by_year["pct_with_disease"], marker="o", color="#c0392b")
        axes[1].set_title("% of articles mentioning a listed disease"); axes[1].set_xlabel("year"); axes[1].set_ylabel("%")
        plt.tight_layout(); plt.show()

**What this shows.** Mean diseases per article rise from about 0.34 in the mid 1990s to a peak near 0.69 around 2022, easing to about 0.63 by 2025. Mean chemicals per article stay roughly flat near 0.10 across the whole period, ending around 0.08, so the growth is on the disease side, not the chemical side. The share of articles mentioning at least one listed disease climbs from about 27 percent to a peak near 46 percent around 2022, settling near 43 percent. These trends reflect both broader entity coverage in abstracts over time and the fixed dictionary term list, so the counts are bounded by that list rather than a complete measure of entity density.

## 6. Summary

- This notebook turns the saved entity tables into actionable analysis: top entities, trends, drug–disease associations, disease co-occurrence communities, and entity richness over time. All analyses read from `data/3_entities/` and can be run with either the dictionary or model-derived tables by changing `METHOD`.
- **Caveats:** dictionary recall is a lower bound; model tables have higher recall but more noise; co-occurrence is association, not causation; counts are document-level and limited to abstracts.

💡 **Next Up:** Proceed to [`11_topic_modeling.ipynb`](11_topic_modeling.ipynb) for unsupervised topic modeling over the abstracts: discovering the latent themes in the corpus and tracking how they shift over time.